# Power Grid Supply-Demand-Congestion Mapper — Analysis Dashboard

This notebook provides an interactive dashboard for exploring how physical grid conditions
connect to market outcomes. Select a region and date range to see which conditions coincided
with high congestion, large price spreads, low renewable output, high load, or outage-heavy periods.

## Data Sources
- **Load data**: EIA hourly demand by balancing authority
- **Generation mix**: EIA hourly generation by fuel type
- **Prices**: LMP with congestion and loss components
- **Outages**: Planned and forced generator outages
- **Weather**: Temperature, wind, precipitation
- **Derived features**: Renewable share, load percentiles, reserve margin proxy, etc.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from config.database import db_config
from config.settings import settings
from etl.load.db_loader import DatabaseLoader
from etl.transform.features import FeatureEngineer
from etl.pipeline import ETLPipeline
from database.seed_sample_data import generate_sample_data
from utils.logging_utils import logger

In [ ]:
# Initialize database loader and feature engineer
db = DatabaseLoader()
fe = FeatureEngineer()

# Try DB connection; fall back to sample CSV if unavailable
DB_AVAILABLE = db.test_connection()
if DB_AVAILABLE:
    print("Database connected — loading from MySQL")
else:
    print("Database not available — generating sample CSV data for demo")
    sample_files = generate_sample_data()
    csv_dir = Path(sample_files.get('hourly_load', 'data/sample')).parent

In [ ]:
def load_data():
    """Load data from DB or CSV fallback."""
    if DB_AVAILABLE:
        try:
            load_df = pd.read_sql("SELECT * FROM hourly_load", db.engine)
            mix_df = pd.read_sql("SELECT * FROM generation_mix", db.engine)
            price_df = pd.read_sql("SELECT * FROM prices", db.engine)
            weather_df = pd.read_sql("SELECT * FROM weather_conditions", db.engine)
            outage_df = pd.read_sql("SELECT * FROM outages", db.engine)
            features_df = pd.read_sql("SELECT * FROM derived_features", db.engine)
            return load_df, mix_df, price_df, weather_df, outage_df, features_df
        except Exception as e:
            print(f"DB load failed: {e}. Falling back to CSV.")

    csv_dir = Path("data/sample")
    load_df = pd.read_csv(csv_dir / "hourly_load.csv", parse_dates=['interval_start_utc', 'interval_end_utc'])
    mix_df = pd.read_csv(csv_dir / "generation_mix.csv", parse_dates=['interval_start_utc', 'interval_end_utc'])
    price_df = pd.read_csv(csv_dir / "prices.csv", parse_dates=['interval_start_utc', 'interval_end_utc'])
    weather_df = pd.read_csv(csv_dir / "weather.csv", parse_dates=['interval_start_utc', 'interval_end_utc'])
    outage_df = pd.read_csv(csv_dir / "outages.csv", parse_dates=['outage_start_utc', 'outage_end_utc'])
    features_df = None
    return load_df, mix_df, price_df, weather_df, outage_df, features_df


load_df, mix_df, price_df, weather_df, outage_df, features_df = load_data()

print(f"Load: {len(load_df):,} rows")
print(f"Generation mix: {len(mix_df):,} rows")
print(f"Prices: {len(price_df):,} rows")
print(f"Weather: {len(weather_df):,} rows")
print(f"Outages: {len(outage_df):,} rows")
print(f"Features: {len(features_df):,} rows" if features_df is not None else "Features: not pre-computed")

In [ ]:
# Build or load the derived feature table
if features_df is None:
    print("Computing derived features...")
    features_df = fe.build_feature_table(load_df, mix_df, price_df, outage_df, weather_df)
print(f"Feature table ready: {len(features_df):,} rows x {len(features_df.columns):,} cols")

---
## Interactive Dashboard

Use the controls below to select a region and date range.

In [ ]:
# --- WIDGETS ---
ba_codes = sorted(features_df['balancing_authority_code'].unique())

region_selector = widgets.Dropdown(
    options=ba_codes,
    value=ba_codes[0] if ba_codes else None,
    description='Region:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)

date_min = features_df['interval_start_utc'].min()
date_max = features_df['interval_start_utc'].max()

date_range = widgets.DateRangeSlider(
    value=(date_min, date_max),
    min=date_min,
    max=date_max,
    step=1,
    description='Date Range:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px'),
)

metric_selector = widgets.Dropdown(
    options=['congestion_spread_max', 'avg_lmp_usd_per_mwh', 'renewable_share',
             'thermal_share', 'load_mw', 'reserve_margin_proxy',
             'total_outage_mw', 'temperature_c'],
    value='congestion_spread_max',
    description='Highlight Metric:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px'),
)

update_btn = widgets.Button(
    description='Update Dashboard',
    button_style='primary',
    layout=widgets.Layout(width='200px'),
)

output = widgets.Output()

display(widgets.VBox([region_selector, date_range, metric_selector, update_btn, output]))

In [ ]:
def build_dashboard(ba_code, start_date, end_date, metric):
    """Build and display the full dashboard for a region and date range."""
    mask = (
        (features_df['balancing_authority_code'] == ba_code)
        & (features_df['interval_start_utc'] >= pd.Timestamp(start_date))
        & (features_df['interval_start_utc'] <= pd.Timestamp(end_date) + timedelta(days=1))
    )
    sub = features_df[mask].sort_values('interval_start_utc').copy()

    if sub.empty:
        with output:
            clear_output(wait=True)
            print(f"No data for {ba_code} in selected range.")
        return

    sub['date'] = sub['interval_start_utc'].dt.date
    daily = sub.groupby('date').agg({
        'load_mw': 'mean',
        'renewable_share': 'mean',
        'thermal_share': 'mean',
        'congestion_spread_max': 'max',
        'congestion_spread_avg': 'mean',
        'avg_lmp_usd_per_mwh': 'mean',
        'reserve_margin_proxy': 'mean',
        'total_outage_mw': 'sum',
        'temperature_c': 'mean',
        'is_extreme_weather': 'max',
    }).reset_index()

    highlight_col = metric

    fig = make_subplots(
        rows=4, cols=2,
        subplot_titles=(
            'Congestion Spread (Max)', 'Average LMP Price',
            'Renewable Share', 'Thermal Generation Share',
            'Hourly Load', 'Reserve Margin Proxy',
            'Outage Capacity', 'Temperature',
        ),
        vertical_spacing=0.08,
        horizontal_spacing=0.1,
    )

    plots = [
        ('congestion_spread_max', 'Congestion Spread ($/MWh)', px.colors.sequential.Reds[-3]),
        ('avg_lmp_usd_per_mwh', 'Avg LMP ($/MWh)', px.colors.sequential.Oranges[-3]),
        ('renewable_share', 'Renewable Share', px.colors.sequential.Greens[-3]),
        ('thermal_share', 'Thermal Share', px.colors.sequential.Blues[-3]),
        ('load_mw', 'Load (MW)', px.colors.sequential.Purples[-3]),
        ('reserve_margin_proxy', 'Reserve Margin', px.colors.sequential.Burg[-3]),
        ('total_outage_mw', 'Outage MW', px.colors.sequential.Greys[-3]),
        ('temperature_c', 'Temperature (°C)', px.colors.sequential.Teal[-3]),
    ]

    for idx, (col, title, color) in enumerate(plots):
        row = idx // 2 + 1
        col_idx = idx % 2 + 1

        is_highlight = (col == highlight_col)
        line_color = 'red' if is_highlight else color
        line_width = 3 if is_highlight else 1.5

        fig.add_trace(
            go.Scatter(
                x=daily['date'], y=daily[col],
                mode='lines+markers',
                name=title,
                line=dict(color=line_color, width=line_width),
                marker=dict(size=4 if not is_highlight else 6),
                showlegend=False,
                hovertemplate=f'%{{x}}<br>{title}: %{{y:.2f}}<extra></extra>',
            ),
            row=row, col=col_idx,
        )

    fig.update_layout(
        height=1000,
        title_text=f'<b>{ba_code} Grid Conditions</b> — {start_date} to {end_date}',
        title_font_size=18,
        hovermode='x unified',
        template='plotly_white',
    )
    fig.update_xaxes(title_text='Date', tickangle=45)
    fig.update_yaxes(title_text='', matches=None)

    with output:
        clear_output(wait=True)
        fig.show()

        # --- Summary statistics ---
        print(f"\n{'='*70}")
        print(f"  {ba_code} | {start_date} to {end_date} | {len(sub):,} hourly records")
        print(f"{'='*70}")

        p99_load = sub['load_mw'].quantile(0.99)
        p95_load = sub['load_mw'].quantile(0.95)
        high_load_hours = sub[sub['load_mw'] >= p95_load]

        print(f"\n  LOAD:")
        print(f"    Avg: {sub['load_mw'].mean():,.0f} MW  |  Peak: {sub['load_mw'].max():,.0f} MW  |  95th: {p95_load:,.0f} MW")

        print(f"\n  GENERATION:")
        print(f"    Avg Renewable Share: {sub['renewable_share'].mean()*100:.1f}%")
        print(f"    Avg Thermal Share:   {sub['thermal_share'].mean()*100:.1f}%")

        top_cong = sub.nlargest(10, 'congestion_spread_max')
        print(f"\n  CONGESTION (top 10 hours):")
        for _, row in top_cong.iterrows():
            ts = row['interval_start_utc'].strftime('%Y-%m-%d %H:00')
            print(f"    {ts}  |  Spread: ${row['congestion_spread_max']:.2f}  |  "
                  f"Load: {row['load_mw']:,.0f} MW  |  "
                  f"Renewable: {row['renewable_share']*100:.1f}%  |  "
                  f"Temp: {row['temperature_c']:.1f}°C")

        print(f"\n  PRICE:")
        print(f"    Avg LMP: ${sub['avg_lmp_usd_per_mwh'].mean():.2f}/MWh  |  "
              f"Max: ${sub['avg_lmp_usd_per_mwh'].max():.2f}/MWh")

        if 'total_outage_mw' in sub.columns and sub['total_outage_mw'].sum() > 0:
            outage_hours = sub[sub['total_outage_mw'] > 0]
            print(f"\n  OUTAGES:")
            print(f"    Hours with outages: {len(outage_hours):,}  |  "
                  f"Total MW lost: {sub['total_outage_mw'].sum():,.0f}")

        if 'is_extreme_weather' in sub.columns:
            extreme_hours = sub[sub['is_extreme_weather'] == True]
            print(f"\n  EXTREME WEATHER:")
            print(f"    Hours with extreme conditions: {len(extreme_hours):,}")

        print(f"\n  CONDITIONS COINCIDING WITH HIGH CONGESTION (top 5%):")
        congestion_threshold = sub['congestion_spread_max'].quantile(0.95)
        high_cong = sub[sub['congestion_spread_max'] >= congestion_threshold]
        if len(high_cong) > 0:
            print(f"    Threshold: ${congestion_threshold:.2f}/MWh")
            print(f"    Avg load: {high_cong['load_mw'].mean():,.0f} MW (vs all-hours avg: {sub['load_mw'].mean():,.0f})")
            print(f"    Avg renewable share: {high_cong['renewable_share'].mean()*100:.1f}% (vs all-hours: {sub['renewable_share'].mean()*100:.1f}%)")
            print(f"    Avg temp: {high_cong['temperature_c'].mean():.1f}°C (vs all-hours: {sub['temperature_c'].mean():.1f}°C)")
            print(f"    Extreme weather fraction: {high_cong['is_extreme_weather'].mean()*100:.1f}% (vs all-hours: {sub['is_extreme_weather'].mean()*100:.1f}%)")
            print(f"    Avg outage MW: {high_cong['total_outage_mw'].mean():.1f} (vs all-hours: {sub['total_outage_mw'].mean():.1f})")
            if 'reserve_margin_proxy' in high_cong.columns:
                print(f"    Avg reserve margin: {high_cong['reserve_margin_proxy'].mean()*100:.1f}% (vs all-hours: {sub['reserve_margin_proxy'].mean()*100:.1f}%)")

        print(f"\n{'='*70}")


def on_update_clicked(b):
    build_dashboard(
        region_selector.value,
        date_range.value[0],
        date_range.value[1],
        metric_selector.value,
    )


update_btn.on_click(on_update_clicked)

# Initial render
if ba_codes:
    build_dashboard(region_selector.value, date_range.value[0], date_range.value[1], metric_selector.value)

---
## Cross-Region Comparison

Examine how multiple ISOs compare on key metrics simultaneously.

In [ ]:
# Cross-region comparison
if len(ba_codes) > 1:
    multi = features_df.groupby(
        [features_df['interval_start_utc'].dt.date, 'balancing_authority_code']
    ).agg({
        'congestion_spread_max': 'max',
        'avg_lmp_usd_per_mwh': 'mean',
        'renewable_share': 'mean',
        'load_mw': 'mean',
        'reserve_margin_proxy': 'mean',
        'total_outage_mw': 'sum',
    }).reset_index().rename(columns={'interval_start_utc': 'date'})

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            'Daily Max Congestion Spread', 'Daily Avg LMP',
            'Daily Avg Renewable Share', 'Daily Avg Load',
            'Daily Avg Reserve Margin', 'Daily Total Outage MW',
        ),
        vertical_spacing=0.1,
        horizontal_spacing=0.1,
    )

    metrics_plots = [
        ('congestion_spread_max', '$/MWh'),
        ('avg_lmp_usd_per_mwh', '$/MWh'),
        ('renewable_share', 'pct'),
        ('load_mw', 'MW'),
        ('reserve_margin_proxy', 'pct'),
        ('total_outage_mw', 'MW'),
    ]

    colors = px.colors.qualitative.Set2
    for idx, (metric, unit) in enumerate(metrics_plots):
        row = idx // 3 + 1
        col = idx % 3 + 1
        for j, ba in enumerate(ba_codes):
            ba_data = multi[multi['balancing_authority_code'] == ba]
            fig.add_trace(
                go.Scatter(
                    x=ba_data['date'], y=ba_data[metric],
                    mode='lines',
                    name=ba,
                    line=dict(width=1.5),
                    legendgroup=ba,
                    showlegend=(idx == 0),
                    hovertemplate=f'%{{x}}<br>{ba}: %{{y:.2f}} {unit}<extra></extra>',
                ),
                row=row, col=col,
            )

    fig.update_layout(
        height=700,
        title_text='<b>Cross-Region Comparison</b>',
        template='plotly_white',
        hovermode='x unified',
    )
    fig.show()
else:
    print("Only one region available — cross-region comparison requires multiple BAs.")

---
## Correlation Analysis

Understand which physical grid conditions correlate with market outcomes.

In [ ]:
# Correlation heatmap between key features
corr_cols = [
    'load_mw', 'renewable_share', 'thermal_share',
    'congestion_spread_max', 'congestion_spread_avg',
    'avg_lmp_usd_per_mwh', 'reserve_margin_proxy',
    'total_outage_mw', 'temperature_c',
]
available = [c for c in corr_cols if c in features_df.columns]
corr_df = features_df[available].dropna()
corr_matrix = corr_df.corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    aspect='auto',
    title='Feature Correlation Matrix',
    width=800,
    height=700,
)
fig.update_layout(template='plotly_white')
fig.show()

print("\nKey correlations with congestion spread:")
if 'congestion_spread_max' in corr_matrix.columns:
    with_cong = corr_matrix['congestion_spread_max'].drop('congestion_spread_max').sort_values(ascending=False)
    for var, val in with_cong.items():
        print(f"  {var:30s}: {val:+.3f}")

print("\nKey correlations with LMP price:")
if 'avg_lmp_usd_per_mwh' in corr_matrix.columns:
    with_lmp = corr_matrix['avg_lmp_usd_per_mwh'].drop('avg_lmp_usd_per_mwh').sort_values(ascending=False)
    for var, val in with_lmp.items():
        print(f"  {var:30s}: {val:+.3f}")

---
## Scenario Analysis

Compare average grid conditions across different regimes.

In [ ]:
# Scenario comparison: high vs low conditions
if 'load_percentile_bin' in features_df.columns:
    scenarios = {
        'High Congestion': features_df['congestion_spread_max'] > features_df['congestion_spread_max'].quantile(0.9),
        'Low Congestion': features_df['congestion_spread_max'] <= features_df['congestion_spread_max'].quantile(0.1),
        'High Load (peak)': features_df['load_percentile_bin'].isin(['peak', 'very_high']),
        'Low Load': features_df['load_percentile_bin'].isin(['very_low', 'low']),
        'High Renewable': features_df['renewable_share'] > features_df['renewable_share'].quantile(0.9),
        'Low Renewable': features_df['renewable_share'] <= features_df['renewable_share'].quantile(0.1),
        'Extreme Weather': features_df['is_extreme_weather'] == True,
        'Heavy Outages': features_df['total_outage_mw'] > features_df['total_outage_mw'].quantile(0.9),
    }

    compare_cols = [
        'load_mw', 'renewable_share', 'thermal_share',
        'congestion_spread_max', 'avg_lmp_usd_per_mwh',
        'reserve_margin_proxy', 'total_outage_mw', 'temperature_c',
    ]
    compare_cols = [c for c in compare_cols if c in features_df.columns]

    rows_data = []
    for scenario, mask in scenarios.items():
        subset = features_df[mask]
        row = {'Scenario': scenario, 'Count': len(subset)}
        for col in compare_cols:
            row[f'Avg {col}'] = subset[col].mean()
        rows_data.append(row)

    scenario_df = pd.DataFrame(rows_data)

    fig = go.Figure()
    for col in compare_cols:
        fig.add_trace(go.Bar(
            name=col,
            x=scenario_df['Scenario'],
            y=scenario_df[f'Avg {col}'],
            hovertemplate='%{x}<br>%{y:.2f}<extra></extra>',
        ))

    fig.update_layout(
        barmode='group',
        title='Scenario Comparison: Average Conditions by Regime',
        template='plotly_white',
        height=500,
        legend_title='Metric',
        xaxis_tickangle=-30,
    )
    fig.show()

    display(scenario_df.style.format({
        'Count': '{:,}',
        **{f'Avg {c}': '{:.2f}' for c in compare_cols},
    }).background_gradient(cmap='RdBu_r', axis=0))
else:
    print("Load percentile bins not available for scenario analysis.")

---
## Generation Mix Composition

Visualize the fuel mix breakdown for the selected region over time.

In [ ]:
# Generation mix stacked area chart for selected region
if not mix_df.empty:
    selected_ba = region_selector.value
    mix_sub = mix_df[
        (mix_df['balancing_authority_code'] == selected_ba)
    ].copy()
    mix_sub['date'] = mix_sub['interval_start_utc'].dt.date

    daily_mix = mix_sub.groupby(['date', 'fuel_type_code'])['generation_mw'].sum().reset_index()

    fig = px.area(
        daily_mix,
        x='date',
        y='generation_mw',
        color='fuel_type_code',
        title=f'{selected_ba} — Daily Generation by Fuel Type',
        labels={'generation_mw': 'Generation (MW)', 'date': 'Date'},
        template='plotly_white',
        height=500,
    )
    fig.update_layout(legend_title='Fuel Type')
    fig.show()
else:
    print("No generation mix data available.")

---
## Outage Timeline

View forced and planned outages over time for the selected region.

In [ ]:
# Outage timeline
if not outage_df.empty:
    selected_ba = region_selector.value
    outage_sub = outage_df[outage_df['balancing_authority_code'] == selected_ba].copy()

    if not outage_sub.empty:
        outage_sub['outage_start_utc'] = pd.to_datetime(outage_sub['outage_start_utc'])
        outage_sub['outage_end_utc'] = pd.to_datetime(outage_sub['outage_end_utc'])
        outage_sub['duration_hours'] = (
            outage_sub['outage_end_utc'] - outage_sub['outage_start_utc']
        ).dt.total_seconds() / 3600

        color_map = {'forced': 'red', 'planned': 'orange', 'maintenance': 'blue', 'other': 'gray'}
        fig = go.Figure()

        for _, row in outage_sub.iterrows():
            fig.add_trace(go.Scatter(
                x=[row['outage_start_utc'], row['outage_end_utc']],
                y=[row['capacity_affected_mw'], row['capacity_affected_mw']],
                mode='lines+markers',
                name=row['outage_type'],
                legendgroup=row['outage_type'],
                showlegend=False,
                line=dict(
                    color=color_map.get(row['outage_type'], 'gray'),
                    width=4,
                ),
                marker=dict(size=6),
                hovertemplate=(f"{row['outage_type'].title()} outage<br>"
                              f"{row['capacity_affected_mw']:.0f} MW<br>"
                              f"Start: %{{x[0]}}<br>"
                              f"Duration: {row['duration_hours']:.0f}h<extra></extra>"),
            ))

        fig.update_layout(
            title=f'{selected_ba} — Outage Timeline',
            xaxis_title='Date',
            yaxis_title='Capacity Affected (MW)',
            template='plotly_white',
            height=400,
            hovermode='closest',
        )
        fig.show()
    else:
        print(f"No outages recorded for {selected_ba}.")
else:
    print("No outage data available.")